In [17]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# training file: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [18]:
# read it in to inspect it
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("length of dataset in characters: ", len(text))
print(text[0])

length of dataset in characters:  1115394
F


In [19]:
chars = sorted(list(set(text)))
print(chars)
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [20]:
# create a simple encode/ decode method
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [
    stoi[c] for c in s
]  # encoder: take a string, output a list of integers
decode = lambda l: "".join(
    [itos[i] for i in l]
)  # decoder: take a list of integers, output a string
print(encode("hii there"))
print(decode(encode("hii there")))


# tokenizer eg: openai's tiktoken

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [21]:
import torch

data = torch.tensor(
    encode(text), dtype=torch.long
)  # create a one dimension tensor (array)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [22]:
# Let's now split up the data into train and validation sets
n = int(0.9 * len(data))  # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [23]:
block_size = 8
train_data[: block_size + 1]  # a chunk of training data set
# each chunk has its context

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [24]:
# spell out the chunk
x = train_data[:block_size]
y = train_data[1 : block_size + 1]
for t in range(block_size):
    context = x[: t + 1]
    target = y[t]
    print(f"when the context is {context} the target is expected to be {target}")

when the context is tensor([18]) the target is expected to be 47
when the context is tensor([18, 47]) the target is expected to be 56
when the context is tensor([18, 47, 56]) the target is expected to be 57
when the context is tensor([18, 47, 56, 57]) the target is expected to be 58
when the context is tensor([18, 47, 56, 57, 58]) the target is expected to be 1
when the context is tensor([18, 47, 56, 57, 58,  1]) the target is expected to be 15
when the context is tensor([18, 47, 56, 57, 58,  1, 15]) the target is expected to be 47
when the context is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is expected to be 58


In [25]:
torch.manual_seed(1337)  # fix the seed to reporoduce the data

batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?


def get_batch(split: '"train" | "x"', batch_size: int = 4, block_size: int = 8):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == "train" else val_data
    ix = torch.randint(
        len(data) - block_size, (batch_size,)
    )  # 0 to data length - block-size
    # print("ix", ix)  # the seed will make ix's result fixed.
    x = torch.stack([data[i : i + block_size] for i in ix])  # add the arbitray offset
    y = torch.stack(
        [data[i + 1 : i + block_size + 1] for i in ix]
    )  # additional 1 offset
    return x, y


xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

print("----")

# b is row number starts from 0
for b in range(batch_size):  # batch dimension
    # t is column number starts from 0, and the training target can be constructed like
    # in each row first x elements from train data will try to perdict the result at the
    # same row and the same x column
    for t in range(block_size):  # time dimension
        context = xb[b, : t + 1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [26]:
import torch
import torch.nn as nn
from torch.nn import functional as F


class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size: int):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx: torch.Tensor, targets: "torch.Tensor | None" = None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  # (BxTxC) batch , time , channel

        if targets == None:
            loss = None
        else:
            # pytorch accepts channel as the second param-> (BxT)xC
            # in this case (4,8,65) ==> (32,65)
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            # (4,8) => 32
            targets = targets.view(B * T)

            # when the result is far away from the target, the cross entropy will grow.
            # reference material https://r23456999.medium.com/%E4%BD%95%E8%AC%82-cross-entropy-%E4%BA%A4%E5%8F%89%E7%86%B5-b6d4cef9189d
            # H(p,q) = -Σ p_i log2(qi)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


# torch.manual_seed(1337)


m = BigramLanguageModel(vocab_size)
# __call__ will automatically trigger forward method
logits, loss = m(xb, yb)  # xb, yb  are stacked chunks
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(5.0364, grad_fn=<NllLossBackward0>)


In [27]:
from torch import Tensor


def generate(model: BigramLanguageModel, idx: Tensor, max_new_tokens: int):
    for _ in range(max_new_tokens):
        # forward pass,  calling `forward` get the logits
        # and the loss will be ignored as we does not provide targets
        logits, loss = model(idx)
        # make (b,t,c) dimension into (b,c) dimension,
        # Bigram model only use current token to perdict next token
        # so that we only need to focus on the last position
        # -1 means the last one on the t(time) dimension
        logits = logits[:, -1, :]
        # apply softmax, convert to the probability distribution
        # dimension (b,c)
        probabilities = F.softmax(logits, dim=1)
        # randomly pick one accorfing to the distribution
        # dimension (b,1)
        idx_next = torch.multinomial(probabilities, num_samples=1)
        # the idx_next is the generated token index, concat to the current token stream
        # idx grows from (b, t) => (b, t+1)
        # in next loop use the concated token tensor as input to perdict next token
        idx = torch.cat([idx, idx_next], dim=1)
    return idx


# random characters generated because currently the model is not trained
# aka: the token_embedding_table


def decode_generated(
    m: BigramLanguageModel,
    input: Tensor = torch.zeros((1, 1), dtype=torch.long),
    max_new_tokens: int = 100,
):
    print(decode(generate(m, idx=input, max_new_tokens=max_new_tokens)[0].tolist()))


decode_generated(m)


lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


In [28]:
# start training bigram
# create a PyTorch optimizer
# nn.Module.parameters will auto matically scans all the attributes and return those
# child class instance of `nn.Module`, in this case is `token_embedding_table`
# `lr` => learning rate, epa
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
# the optimizer works like the `micrograd`, it is responsibile for the grad descent,
# but in a smarter way (adjust each paramter's learning rate)

In [29]:
def training_bigram(m: BigramLanguageModel = BigramLanguageModel(vocab_size)):
    xb, yb = get_batch("train", batch_size=32)
    optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

    for _ in range(10000):  # increase number of steps for good results...
        # evaluate the loss
        logits, loss = m(xb, yb)
        # always remember to set to zero!
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    print(loss.item())
    return m


# trained_m = training_bigram(m)
# torch.manual_seed(1337)

training_bigram()

1.6959125995635986


BigramLanguageModel(
  (token_embedding_table): Embedding(65, 65)
)

In [30]:
decode_generated(training_bigram(), max_new_tokens=500)

1.7200144529342651

Agowit tothre go bed, theancade to w dieied, f mit witie Gr'l owisthe! auth
Th t owintheno d? gowill I berudre I bue
A cherow willllie Lad nowo gaieiseie, d,
Spso aust charin, I'l wo me not e!
A wiera-besedee mitr'l ne thil Lofl bedy t;
Wis l thrue I'dith I'sed, s gll w s flllese can, I'dy wo t;
Th t ndind, Gr'l a me kne are ntrrowine!
Therititruthinof a authenow wiliedy d dindy thisowit m I'l trotllis denat thinodrerdise my Of ne ad!
A ano I chue,
Thus s I t wit;
Thast wind caitresthud? wis as 


## The mathematical trick in self-attention

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2  # batch, time, channels
x = torch.randn(B, T, C)
x.shape

# token should only talk to those tokens that  occurs before it, not after it,
# to follow the time sequence


def make_data():
    torch.manual_seed(1337)
    B, T, C = 4, 8, 2  # batch, time, channels
    x = torch.randn(B, T, C)
    x.shape
    return x, B, T, C

In [15]:
# We want x[b,t] = mean_{i<=t} x[b,i]

# bow stands for `back of words`
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, : t + 1]  # (t,C) previou chunk
        xbow[b, t] = torch.mean(xprev, 0)  # make average value
print(xbow)

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])


In [7]:
# instead of using the nested loop, try the matrix multiply by using the trilow
import torch


def matrix_mul():
    torch.manual_seed(42)
    a = torch.tril(torch.ones(3, 3))

    # make average
    d = torch.sum(a, 1, keepdim=True)
    print(d)
    a /= d
    b = torch.randint(0, 10, (3, 2)).float()
    c = (
        a @ b
    )  # `@` is the operator for thea matrix multiplication, built in method is `__matmul__`

    print(a)
    print(b)
    print(c)


matrix_mul()

tensor([[1.],
        [2.],
        [3.]])
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# version 2
def advanced_averaging():
    x, B, T, C = make_data()
    weight = torch.tril(torch.ones(T, T))
    weight /= torch.sum(weight, 1, keepdim=True)
    # xbow2 = wei @ x
    # wei: (T, T), x: (B, T, C) -> xbow2: (B, T, C)
    # @ operates on last two dims, B is batched automatically
    # assert wei.shape == (T, T) and x.shape == (B, T, C)
    xbow2 = weight @ x  # auto batching results (B,T,C)
    # May return False due to floating point precision differences
    # Different computation paths produce tiny numerical errors (e.g. 1e-7)
    # Use atol/rtol tolerance: torch.allclose(xbow, xbow2, atol=1e-6)
    print(torch.allclose(xbow, xbow2, atol=1e-6))


advanced_averaging()

True
tensor([[ 1.6455, -0.8030],
        [ 1.4985, -0.5395],
        [ 0.4954,  0.3420],
        [ 1.0623, -0.1802],
        [ 1.1401, -0.4462],
        [ 1.0870, -0.4071],
        [ 1.0430, -0.1299],
        [ 1.1138, -0.1641]])
tensor([[ 1.6455, -0.8030],
        [ 1.4985, -0.5395],
        [ 0.4954,  0.3420],
        [ 1.0623, -0.1802],
        [ 1.1401, -0.4462],
        [ 1.0870, -0.4071],
        [ 1.0430, -0.1299],
        [ 1.1138, -0.1641]])


In [25]:
from torch.nn import functional as F


# version 3 use softmax
# Masked attention: each token can only attend to itself and past tokens
# prevents the model from "seeing the future" during training
def with_softmax():
    x, B, T, C = make_data()
    tril = torch.tril(torch.ones(T, T))
    weight = torch.zeros((T, T))
    # weight: (T, T) attention weights matrix
    # tril == 0 identifies upper triangle (future tokens)
    # masked_fill with -inf instead of 0 because:
    #   softmax(0) != 0, but softmax(-inf) = e^(-inf) = exactly 0
    #   ensuring future tokens contribute nothing to the weighted sum
    weight = weight.masked_fill(tril == 0, float("-inf"))
    weight = F.softmax(weight, dim=-1)
    # softmax normalizes each row to sum to 1
    # -inf positions become exactly 0 after softmax
    # result: each token only attends to itself and past tokens
    print(weight)
    xbow3 = weight @ x
    print(torch.allclose(xbow, xbow3, atol=1e-6))

    return xbow3


with_softmax()

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
True


tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

Notes:

- Attention is a communication mechanism. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with tril, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [34]:
import torch
import torch.nn as nn


# self-attention mechanism
def self_attention_single_head():
    torch.manual_seed(1337)
    B, T, C = 4, 8, 32  # batch, time, channels
    x = torch.randn(B, T, C)

    # let's see a single Head perform self-attention
    head_size = 16
    query = nn.Linear(C, head_size, bias=False)  # what information do I need
    key = nn.Linear(C, head_size, bias=False)  # what kind of information can I provide
    value = nn.Linear(
        C, head_size, bias=False
    )  # what kind of information I acctually carry
    k = key(x)  # (B, T, 16)
    q = query(x)  # (B, T, 16)
    # the result of q @ k.t is bigger means the information provided is exactly query needs
    wei = q @ k.transpose(-2, -1)  # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

    # scaled dot-product attention
    # q @ k.T produces dot products whose variance grows with head_size
    # if Q, K elements ~ N(0,1), dot product variance = head_size, std = √head_size
    # large values cause softmax to saturate:
    #   softmax([1,2,3])    -> [0.09, 0.24, 0.67]  diffuse
    #   softmax([10,20,30]) -> [0.00, 0.00, 1.00]  collapsed to one token
    # dividing by √head_size normalizes variance back to 1
    # keeps softmax output diffuse, gradients stable during training
    wei = wei * head_size**-0.5

    tril = torch.tril(torch.ones(T, T))
    # wei = torch.zeros((T,T))
    wei = wei.masked_fill(tril == 0, float("-inf"))
    wei = F.softmax(wei, dim=-1)

    v = value(x)
    out = wei @ v
    # out = wei @ x

    print(out.shape)


self_attention_single_head()

torch.Size([4, 8, 16])


In [35]:
import torch.nn as nn
import torch


class Head(nn.Module):
    def __init__(
        self,
        n_embed: int,
        head_size: int,
        block_size: int,
    ):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        # register as a matrix that does not need grad descent, but still involves the
        # training process
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x: torch.Tensor):
        B, T, C = x
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.trill[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        out = wei @ v
        return out

In [ ]:
# attention impls
import torch.nn as nn
import torch
from torch import Tensor

device = "cuda" if torch.cuda.is_available() else "cpu"


class BigramWithAttention(nn.Module):
    def __init__(self, vocab_size: int, n_embed: int, block_size: int, head_size: int):
        # n_embed: dimension of the internal representation space
        # decoupled from vocab_size unlike the Bigram model where embedding dim == vocab_size
        # in Bigram: embedding table directly produced logits (no semantic meaning)
        # in Transformer: embedding encodes semantic features, participates in attention,
        #   accumulates context through multiple layers
        # lm_head then projects n_embed -> vocab_size at the end
        # value is a hyperparameter, typically a power of 2 (32, 64, 128...)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        # lm_head: linear projection from embedding space to vocabulary space
        # (B, T, n_embed) -> (B, T, vocab_size)
        # embedding encodes semantic features, not character probabilities directly
        # this layer converts those features into logits over the vocabulary
        # so the linear represents the matrix (x_embed, vocab_size)

        # nn.Linear(n_embed, vocab_size)
        # weight shape: (vocab_size, n_embed) — weight[i] = weights of i-th output neuron
        # forward: output = input @ weight.T + bias
        # weight is stored transposed by convention, .T restores it for matrix multiply
        # (B, T, n_embed) @ (n_embed, vocab_size) -> (B, T, vocab_size)
        self.lm_head = nn.Linear(n_embed, vocab_size)

        # each row = a learned vector for that position in the sequence
        # necessary because attention is order-agnostic by itself
        #   "A at position 1" and "A at position 5" are indistinguishable without this
        # position embeddings are added to token embeddings:
        #   x = tok_emb + pos_emb  # (B, T, n_embed)
        # hard limit: model can only attend to block_size tokens at a time
        #   tokens beyond block_size are completely invisible to the model
        #   this is what "context window" means in GPT models
        self.position_embedding_table = nn.Embedding(block_size, n_embed)

        self.sa_head = Head(n_embed=n_embed, head_size=head_size, block_size=block_size)

    def forward(self, idx: Tensor, targets: Tensor = None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)  #  (B , T , e_embed)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )  # (T,n_embed)
        x = tok_emb + pos_emb  # auto batching (B,T,n_embed)+(T,n_embed) = (B,T,n_embed)
        x = self.sa_head(x)

        logits: Tensor = self.lm_head(tok_emb)  # (B , T , vocab_size)
        if target == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

In [ ]:
import torch

# detect gpu training is avaliable


def get_batch_cuda(split: str, batch_size: int = 4, block_size: int = 8):
    x, y = get_batch(split, batch_size, block_size)
    x, y = x.to(device), y.to(device)
    return x, y


# tell the pytorch, the following function will not make any grad change
# the kernel will apply performance improvements
@torch.no_grad()
def estimate_loss(
    model: BigramWithAttention,
    eval_iters: int,
    batch_size: int = 4,
    block_size: int = 8,
):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_cuda(split, batch_size, block_size)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


def training_attention(
    model: any,
    max_iters: int,
    eval_interval: int,
    batch_size: int = 4,
    block_size: int = 8,
):
    xb, yb = get_batch_cuda("train", batch_size, block_size)
    # model = BigramWithAttention()
    m = model.to(device)
    optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
    for iter in range(max_iters):
        # every once in a while evaluate the loss on train and val sets
        if iter % eval_interval == 0:
            losses = estimate_loss()
            print(
                f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}"
            )

        # evaluate the loss
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()


def generate_cude(m: BigramWithAttention):
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    decode_generated(m, context, max_new_tokens=500)